In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import pathlib
from datetime import datetime
import json

from src.plot import Plotter
from src.ema import EMA
from src.dataset import get_dataset
from src.model_utils import build_model, build_diffuser, load_config


## 条件付き拡散モデルの学習

In [ ]:
dataset_name = "CIFAR10"  # "MNIST", "FashionMNIST", "CIFAR10"
pred_noise_model_name = "SimpleU-net"  # "SimpleU-net", "Dit"

In [ ]:
DATASET_CONFIG = {
    "MNIST": {
        "channels": 1,
        "num_labels": 10,
        "image_size": 28,
        "image_range": "-1,1",
    },
    "FashionMNIST": {
        "channels": 1,
        "num_labels": 10,
        "image_size": 28,
        "image_range": "-1,1",
    },
    "CIFAR10": {
        "channels": 3,
        "num_labels": 10,
        "image_size": 32,
        "image_range": "-1,1",
    },
}
dataset_config = DATASET_CONFIG[dataset_name]

model_config_all = {
    "SimpleU-net": {
        "model_class": "CondSimpleUnet",
        "model_params": { 
                "in_ch": dataset_config["channels"],
                "time_embed_dim": 100,
                "num_labels": dataset_config["num_labels"]
        },
    },
    "Dit": {
        # TODO: ここにDitのモデル構成を追加
        "model_class": "Dit",  # Ditのモデルクラスを指定
        "model_params": {
            "in_ch": dataset_config["channels"],
            "time_embed_dim": 100,
            "num_labels": dataset_config["num_labels"],
            # Ditのモデルパラメータを指定
        },
    }
}
model_config = model_config_all[pred_noise_model_name]

diffuser_config = {
    "num_timesteps": 500,
    "beta_start": 0.0001,
    "beta_end": 0.02,
    "gamma": 3.0,
    "beta_schedule_type": "cosine",
}

train_config = {
    "dataset_name": dataset_name,
    "dataset_config": dataset_config,
    "batch_size": 128,
    "epochs": 10,
    "lr": 5e-4,
    "without_cond_prob": 0.1,
    "ema_decay": 0.999,
    "save_checkpoint": True,
    "every_n_epochs": 2,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

plotter = Plotter(dataset_name)

In [ ]:
# モデルフォルダの作成
model_name = f"{dataset_name}_{pred_noise_model_name}_{datetime.now().strftime('%Y%m%d')}"
model_dir = pathlib.Path("models")
model_dir.mkdir(exist_ok=True)
model_folder_dir = model_dir / f"{model_name}"
model_folder_dir.mkdir(exist_ok=True)

# Checkpointのパス
checkpoint_path = model_folder_dir / "checkpoints"
checkpoint_path.mkdir(exist_ok=True)
# 学習ウェイトのパス
weights_path = model_folder_dir / "weights"
weights_path.mkdir(exist_ok=True)

# configの保存
config_dir = model_folder_dir / "config"
config_dir.mkdir(exist_ok=True)
with open(config_dir / "model_config.json", "w") as f:
    json.dump(model_config, f)
with open(config_dir / "diffuser_config.json", "w") as f:
    json.dump(diffuser_config, f)
with open(config_dir / "train_config.json", "w") as f:
    json.dump(train_config, f)

In [ ]:
# データセットの準備
dataset = get_dataset(dataset_name, root=pathlib.Path("."), train=True, image_range=dataset_config["image_range"])
dataloader = DataLoader(dataset, batch_size=train_config["batch_size"], shuffle=True)

In [ ]:
# デバッグモードの設定
DEBUG = False
if DEBUG:
    train_config["epochs"] = 2
    diffuser_config["num_timesteps"] = 10
    train_config["batch_size"] = 8
    train_config["every_n_epochs"] = 1

    # debug用にデータセットを小さくする
    dataset = torch.utils.data.Subset(dataset, list(range(1000)))
    dataloader = DataLoader(dataset, batch_size=train_config["batch_size"], shuffle=True, num_workers=0, drop_last=True)

In [ ]:
# 拡散モデルの準備
model = build_model(model_config, device)
diffuser = build_diffuser(model, diffuser_config, device)
# EMAモデルの準備
ema = EMA(model, decay=train_config["ema_decay"])
diffuse_ema = build_diffuser(ema.ema_model, diffuser_config, device)
# Optimizerの準備
optimizer = Adam(diffuser.noise_pred_model.parameters(), lr=train_config["lr"])

In [ ]:
losses = []

for epoch in range(train_config["epochs"]):
    loss_sum = 0
    cnt = 0

    # サンプリングして生成画像をプロット
    labels_test = torch.arange(0, 10).repeat(2)

    images = diffuser.ddim_sampling(
        x_shape=(20, 3, 32, 32),
        labels=labels_test
    )
    plotter.show_images_cond(images, labels_test)

    images = diffuse_ema.ddim_sampling(
        x_shape=(20, 3, 32, 32),
        labels=labels_test
    )
    plotter.show_images_cond(images, labels_test)

    pbar = tqdm(
        dataloader,
        desc=f"Epoch {epoch+1}/{train_config['epochs']}"
    )

    for images, labels in pbar:

        x = images.to(device)
        labels = labels.to(device)

        timestep = torch.randint(
            1,
            diffuser_config["num_timesteps"] + 1,
            (len(x),),
            device=device
        )

        # 分類器なしガイダンス
        if np.random.random() < train_config["without_cond_prob"]:
            labels = None

        x_noisy, noise = diffuser.add_noise(x, timestep)
        noise_pred = diffuser.noise_pred_model(
            x_noisy,
            timestep,
            labels
        )

        loss = F.mse_loss(noise, noise_pred)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        ema.update(model)

        loss_sum += loss.item()
        cnt += 1

        # tqdm に loss 表示
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "avg": f"{loss_sum/cnt:.4f}"
        })

    loss_avg = loss_sum / cnt
    losses.append(loss_avg)
    if train_config["save_checkpoint"] and (epoch + 1) % train_config["every_n_epochs"] == 0:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "ema_model_state_dict": ema.ema_model.state_dict(),
        }, checkpoint_path / f"checkpoint_epoch_{epoch+1:03d}.pth")

In [ ]:
# plot losses
plotter.plot_loss(losses)

## 条件付き拡散モデルによる生成

In [ ]:
# generate samples
labels = torch.arange(0, 10).repeat(2)
images = diffuser.ddpm_sampling(x_shape=(20, 3, 32, 32), labels=labels)
plotter.show_images_cond(images, labels)
images_ema = diffuse_ema.ddpm_sampling(x_shape=(20, 3, 32, 32), labels=labels)
plotter.show_images_cond(images_ema, labels)

In [ ]:
model_final_path = weights_path / "model_final.pth"
model_ema_final_path = weights_path / "model_ema_final.pth"
torch.save(diffuser.noise_pred_model.state_dict(), model_final_path )
torch.save(ema.ema_model.state_dict(), model_ema_final_path )

## 学習済みモデルでの動作検証

In [ ]:
print(f"Model name: {model_name}")

In [ ]:
model_type = "ema"  # "final", "ema"

if model_type == "final":
    model_path = model_final_path
elif model_type == "ema":
    model_path = model_ema_final_path

model_config_test, diffuser_config_test, train_config_test = load_config(model_name)

model_test = build_model(model_config_test, device)
model_test.load_state_dict(torch.load(model_path))
diffuser_test = build_diffuser(model_test, diffuser_config_test, device)

In [ ]:
# generate samples
labels = torch.arange(0, 10).repeat(2)
images = diffuser_test.ddim_sampling(x_shape=(20, 3, 32, 32), labels=labels)
plotter.show_images_cond(images, labels)

In [ ]:
# generate samples
# labels = torch.arange(0, 10).repeat(2)
# images = diffuser_test.ddpm_sampling(x_shape=(20, 3, 32, 32), labels=labels)
# plotter.show_images_cond(images, labels)